# TransitPulse: transformer demand forecaster on 1B+ real NYC trips

Trains a temporal transformer for multi-zone day-ahead demand forecasting on the NYC TLC high-volume for-hire-vehicle corpus (every Uber/Lyft trip since Feb 2019, ~20M/month).

Setup: Kaggle notebook, GPU on (T4/P100), Internet on.

Protocol matches the TransitPulse repo: strict temporal holdout (last 90 days), seasonal-naive baseline (same hour, 7 days earlier), Poisson likelihood on raw counts. Report both numbers, always.

In [ ]:
%pip install -q duckdb

# Config. TEST_MODE trains on 6 months for a quick smoke run; flip it off
# for the full corpus (Kaggle disk holds ~55 GB in /kaggle/working).
TEST_MODE = False
START, END = ("2025-01", "2025-06") if TEST_MODE else ("2021-01", "2026-04")
INPUT_HOURS = 336      # two weeks of history per sample
HORIZON = 24           # predict the next day, hourly
VALID_DAYS = 90
D_MODEL, N_HEADS, N_LAYERS = 128, 8, 4
BATCH, EPOCHS, LR = 256, 12, 3e-4

In [ ]:
import os, time, requests
from pathlib import Path

RAW = Path("/kaggle/working/raw"); RAW.mkdir(exist_ok=True)
CDN = "https://d37ci6vzurychx.cloudfront.net/trip-data"

def months(a, b):
    y, m = map(int, a.split("-")); y2, m2 = map(int, b.split("-"))
    while (y, m) <= (y2, m2):
        yield f"{y:04d}-{m:02d}"
        y, m = (y + 1, 1) if m == 12 else (y, m + 1)

for mo in months(START, END):
    dest = RAW / f"fhvhv_tripdata_{mo}.parquet"
    if dest.exists():
        continue
    r = requests.get(f"{CDN}/{dest.name}", stream=True, timeout=300)
    if r.status_code in (403, 404):
        print(mo, "not published, stopping"); break
    with open(dest, "wb") as f:
        for chunk in r.iter_content(1 << 22):
            f.write(chunk)
    print(mo, f"{dest.stat().st_size/1e6:.0f} MB", flush=True)
print("on disk:", sum(f.stat().st_size for f in RAW.glob('*.parquet'))/1e9, "GB")

In [ ]:
import duckdb

con = duckdb.connect()
panel = con.execute(f"""
    SELECT CAST(pickup_datetime AS DATE) AS date,
           EXTRACT(hour FROM pickup_datetime) AS hour,
           PULocationID AS zone, COUNT(*) AS trips
    FROM read_parquet('{RAW.as_posix()}/fhvhv_tripdata_*.parquet',
                      union_by_name=true)
    GROUP BY ALL ORDER BY zone, date, hour
""").df()
print(f"{len(panel):,} panel rows, {panel.trips.sum():,} trips, "
      f"{panel.zone.nunique()} zones, {panel.date.min()} to {panel.date.max()}")

In [ ]:
import numpy as np, pandas as pd

# Dense [zone, hour-index] count matrix on a continuous hourly clock
panel["ts"] = pd.to_datetime(panel.date) + pd.to_timedelta(panel.hour, unit="h")
clock = pd.date_range(panel.ts.min(), panel.ts.max(), freq="h")
t_index = pd.Series(np.arange(len(clock)), index=clock)
zones = np.sort(panel.zone.unique())
z_index = {z: i for i, z in enumerate(zones)}

M = np.zeros((len(zones), len(clock)), dtype=np.float32)
M[panel.zone.map(z_index).to_numpy(), t_index[panel.ts].to_numpy()] = \
    panel.trips.to_numpy(dtype=np.float32)

hours = clock.hour.to_numpy(); dows = clock.dayofweek.to_numpy()
CAL = np.stack([np.sin(2*np.pi*hours/24), np.cos(2*np.pi*hours/24),
                np.sin(2*np.pi*dows/7),  np.cos(2*np.pi*dows/7)],
               axis=1).astype(np.float32)

valid_start = len(clock) - VALID_DAYS * 24
# Sample start indices: one window per zone per day. A window belongs to
# validation when its prediction target lies inside the holdout.
starts = np.arange(0, len(clock) - INPUT_HOURS - HORIZON, 24)
train_starts = starts[starts + INPUT_HOURS + HORIZON <= valid_start]
valid_starts = starts[starts + INPUT_HOURS >= valid_start]
print(f"zones {len(zones)}, hours {len(clock)}, "
      f"train windows {len(train_starts)*len(zones):,}, "
      f"valid windows {len(valid_starts)*len(zones):,}")

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class Windows(Dataset):
    def __init__(self, starts):
        self.pairs = [(z, s) for z in range(len(zones)) for s in starts]
    def __len__(self):
        return len(self.pairs)
    def __getitem__(self, i):
        z, s = self.pairs[i]
        x = M[z, s:s+INPUT_HOURS]
        y = M[z, s+INPUT_HOURS:s+INPUT_HOURS+HORIZON]
        cal = CAL[s:s+INPUT_HOURS]
        feats = np.concatenate([np.log1p(x)[:, None], cal], axis=1)
        return feats, np.int64(z), y

class DemandTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(5, D_MODEL)
        self.zone_emb = nn.Embedding(len(zones), D_MODEL)
        self.pos = nn.Parameter(torch.randn(1, INPUT_HOURS, D_MODEL) * 0.02)
        layer = nn.TransformerEncoderLayer(
            D_MODEL, N_HEADS, 4*D_MODEL, dropout=0.1,
            batch_first=True, norm_first=True)
        self.enc = nn.TransformerEncoder(layer, N_LAYERS)
        self.head = nn.Sequential(nn.LayerNorm(D_MODEL),
                                  nn.Linear(D_MODEL, HORIZON))
    def forward(self, feats, z):
        h = self.proj(feats) + self.pos + self.zone_emb(z)[:, None, :]
        h = self.enc(h).mean(dim=1)
        return nn.functional.softplus(self.head(h)) + 1e-6  # Poisson rates

dev = "cuda" if torch.cuda.is_available() else "cpu"
model = DemandTransformer().to(dev)
print(sum(p.numel() for p in model.parameters())/1e6, "M params, device:", dev)

In [ ]:
train_dl = DataLoader(Windows(train_starts), batch_size=BATCH, shuffle=True,
                      num_workers=2, pin_memory=True)
valid_dl = DataLoader(Windows(valid_starts), batch_size=BATCH,
                      num_workers=2, pin_memory=True)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS*len(train_dl))
scaler = torch.amp.GradScaler()
nll = nn.PoissonNLLLoss(log_input=False, full=False)

best = float("inf")
for ep in range(EPOCHS):
    model.train(); t0 = time.time(); tot = 0.0
    for feats, z, y in train_dl:
        feats, z, y = feats.to(dev), z.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast(dev):
            loss = nll(model(feats, z), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item() * len(z)
    model.eval(); vtot, n = 0.0, 0
    with torch.no_grad():
        for feats, z, y in valid_dl:
            feats, z, y = feats.to(dev), z.to(dev), y.to(dev)
            vtot += nll(model(feats, z), y).item() * len(z); n += len(z)
    v = vtot / n
    print(f"epoch {ep}: train {tot/len(train_dl.dataset):.4f} "
          f"valid {v:.4f} ({time.time()-t0:.0f}s)", flush=True)
    if v < best:
        best = v
        torch.save(model.state_dict(), "/kaggle/working/demand_transformer.pt")

In [ ]:
# Evaluate vs seasonal-naive (same hour one week earlier) on the holdout
model.load_state_dict(torch.load("/kaggle/working/demand_transformer.pt"))
model.eval()
preds, actuals, naives = [], [], []
with torch.no_grad():
    for feats, z, y in valid_dl:
        p = model(feats.to(dev), z.to(dev)).cpu().numpy()
        preds.append(p); actuals.append(y.numpy())
        x = np.expm1(feats[:, :, 0].numpy())
        naives.append(x[:, INPUT_HOURS-168:INPUT_HOURS-168+HORIZON])
pred = np.concatenate(preds).ravel(); act = np.concatenate(actuals).ravel()
nai = np.concatenate(naives).ravel()

def r2(y, p):
    return 1 - ((y-p)**2).sum() / ((y-y.mean())**2).sum()
import json
metrics = {
    "model_r2": round(float(r2(act, pred)), 4),
    "model_mae": round(float(np.abs(act-pred).mean()), 3),
    "naive_r2": round(float(r2(act, nai)), 4),
    "naive_mae": round(float(np.abs(act-nai).mean()), 3),
    "n_valid_windows": int(len(np.concatenate(actuals))),
    "zones": int(len(zones)), "train_months": f"{START}..{END}",
}
json.dump(metrics, open("/kaggle/working/transformer_metrics.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))

Download `demand_transformer.pt` and `transformer_metrics.json` from the notebook output and commit the metrics to the repo (`kaggle/results/`). Never quote the model number without the naive number next to it.